In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, confusion_matrix, precision_score, recall_score
import joblib
import warnings
warnings.filterwarnings('ignore')

print("All imports loaded")

All imports loaded


In [2]:
fraud_df = pd.read_csv('../data/processed/fraud_data_processed.csv')
print(f"Shape: {fraud_df.shape}")
print(f"Fraud rate: {fraud_df['class'].mean()*100:.4f}%")

Shape: (151112, 20)
Fraud rate: 9.3646%


In [3]:
categorical_cols = ['source', 'browser', 'sex', 'country']
fraud_df = pd.get_dummies(fraud_df, columns=categorical_cols, drop_first=True)

feature_cols = ['purchase_value', 'age', 'hour_of_day', 'day_of_week', 
                'time_since_signup_hours', 'new_user_1h', 'users_per_device']
feature_cols += [col for col in fraud_df.columns if col.startswith(('source_', 'browser_', 'sex_', 'country_'))]

X = fraud_df[feature_cols].fillna(0)
y = fraud_df['class']

print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")

Features: 14
Samples: 151112


In [4]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")
print(f"Train fraud rate: {y_train.mean()*100:.4f}%")

Train: 120889 samples
Test: 30223 samples
Train fraud rate: 9.3648%


In [5]:
print("BEFORE SMOTE:")
print(f"  Class 0: {sum(y_train==0)}")
print(f"  Class 1: {sum(y_train==1)}")

smote = SMOTE(random_state=42, sampling_strategy=0.5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("\nAFTER SMOTE:")
print(f"  Class 0: {sum(y_train_resampled==0)}")
print(f"  Class 1: {sum(y_train_resampled==1)}")

BEFORE SMOTE:
  Class 0: 109568
  Class 1: 11321

AFTER SMOTE:
  Class 0: 109568
  Class 1: 54784


In [6]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, scale_pos_weight=5, eval_metric='logloss')
}

results = {}
print("Training models...\n")

for name, model in models.items():
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    results[name] = {'model': model, 'f1': f1}
    print(f"{name}: F1-Score = {f1:.4f}")

Training models...

Logistic Regression: F1-Score = 0.6117
Random Forest: F1-Score = 0.6193
XGBoost: F1-Score = 0.6154


In [7]:
print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)

for name, result in sorted(results.items(), key=lambda x: x[1]['f1'], reverse=True):
    print(f"{name:<20} F1: {result['f1']:.4f}")

best_name = max(results, key=lambda x: results[x]['f1'])
best_f1 = results[best_name]['f1']
print(f"\n BEST MODEL: {best_name} (F1 = {best_f1:.4f})")


MODEL COMPARISON
Random Forest        F1: 0.6193
XGBoost              F1: 0.6154
Logistic Regression  F1: 0.6117

 BEST MODEL: Random Forest (F1 = 0.6193)


In [8]:
joblib.dump(results[best_name]['model'], '../models/best_fraud_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
print(" Model saved to ../models/best_fraud_model.pkl")
print(" Scaler saved to ../models/scaler.pkl")

 Model saved to ../models/best_fraud_model.pkl
 Scaler saved to ../models/scaler.pkl
